# 03 — Run scenarios

Runs three comparable scenarios on the same agent population and writes per-agent / per-station / per-scenario CSVs plus the required figures.

Scenarios:
1. **S1_real** — real OCM/EIPA charger layout.
2. **S2_clustered** — same number of stations and total ports, clustered near the city centre.
3. **S3_distributed** — same number of stations and total ports, on a coarse grid covering Śródmieście.

Same RNG seed, same agent count, same horizon → differences in metrics are attributable to layout only.

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

from src import config, graph_utils, charger_data, agents as agents_mod, scenarios as scen_mod, simulation as sim_mod, metrics as metrics_mod

In [ ]:
G = graph_utils.get_or_build_graph()
df_clean = charger_data.load_clean_chargers()
print(f'graph: {G.number_of_nodes():,} nodes  | clean chargers: {len(df_clean)}')

In [ ]:
agents = agents_mod.generate_agents(G)
print(f'generated {len(agents)} agents, total planned trips = {sum(len(a.trips) for a in agents)}')

In [ ]:
import copy
scenarios = scen_mod.build_all_scenarios(df_clean, G)
for s in scenarios:
    print(f'{s.name}: {len(s.stations)} stations | total ports = {s.stations.total_ports()}')

In [ ]:
# IMPORTANT: each scenario gets a *fresh deep copy* of the agent list so that
# state from one scenario does not leak into the next.
results = []
for s in scenarios:
    fresh_agents = copy.deepcopy(agents)
    sim = sim_mod.Simulator(G, fresh_agents, s.stations, scenario_name=s.name)
    results.append(sim.run(progress=True))

In [ ]:
import pandas as pd
pd.DataFrame([r.summary for r in results])

In [ ]:
paths = metrics_mod.write_all_outputs(results)
for k, v in paths.items():
    print(f'{k:36s} → {v}')